# 02 — Modélisation

**Objectif :** entraîner, comparer et évaluer les modèles. Tout le code est dans `src/train.py` et `src/evaluate.py` ; ce notebook l'exécute et commente les résultats.

Démarche :
1. découpage **par joueur** en entraînement (60 %), validation (20 %) et test (20 %) ;
2. deux candidats : **régression logistique** (référence simple) et **gradient boosting** ;
3. choix du modèle sur la validation (meilleure PR-AUC) ;
4. choix du **seuil de décision** : le plus bas qui garde ≤ 1 % de faux positifs ;
5. évaluation **unique** sur le test, comparée à la règle actuelle (signalements).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Permet d'importer le code du dossier src/ depuis le dossier notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.viz import BLUE, GRAY, HONEST_CHEATER_LABELS, ORANGE, apply_style

apply_style()
pd.set_option("display.precision", 3)

In [ ]:
from src.config import FEATURES, TARGET
from src.data.generate import generate_dataset
from src.train import split_by_player, train_and_select

df = generate_dataset(n_matches=50_000, seed=42)
train, val, test = split_by_player(df, seed=42)
print(f"Entraînement : {len(train):,} | Validation : {len(val):,} | Test : {len(test):,}")
print(
    "Joueurs communs entre entraînement et test :",
    len(set(train["player_id"]) & set(test["player_id"])),
)

**Pourquoi découper par joueur ?** Un joueur joue plusieurs matchs. Si certains de ses matchs étaient dans l'entraînement et d'autres dans le test, le modèle reconnaîtrait le joueur (même ancienneté, même rang…) au lieu de détecter la triche : les scores seraient trop optimistes. C'est une **fuite de données**. Résultat ci-dessus : 0 joueur commun.

In [ ]:
result = train_and_select(df, seed=42)
columns = ["pr_auc", "roc_auc", "precision_top", "recall", "false_positive_rate"]
pd.DataFrame(result.validation_metrics).T[columns]

**Choix du modèle :** les deux candidats sont très proches. Le gradient boosting est retenu pour sa PR-AUC légèrement supérieure et parce qu'il capte les **combinaisons** de signaux (ex. précision élevée *et* réaction très rapide), que la régression logistique ne voit qu'additionnées. Son explicabilité est assurée par SHAP (notebook 03).

In [ ]:
comparison = pd.DataFrame(
    {"Sentinel": result.test_metrics, "Règle des signalements": result.baseline_test_metrics}
).loc[["pr_auc", "roc_auc", "precision_top", "recall", "false_positive_rate", "threshold"]]
comparison

**Lecture des métriques (jeu de test) :**
- `precision_top` : part de vrais tricheurs dans le **1 % le plus suspect** (≈ les 300 dossiers examinés chaque jour).
- `recall` : part des tricheurs détectés au seuil de décision.
- `false_positive_rate` : part des joueurs **honnêtes** signalés à tort (plafonnée à ~1 %).
- `pr_auc` : qualité globale du classement, adaptée aux classes déséquilibrées.

In [ ]:
from sklearn.metrics import precision_recall_curve

test_scores = result.model.predict_proba(test[FEATURES])[:, 1]
y_test = test[TARGET].to_numpy()

fig, ax = plt.subplots(figsize=(7, 4.5))
for scores, label, color in [
    (test_scores, "Sentinel", BLUE),
    (test["reports_received"].to_numpy(dtype=float), "Règle des signalements", ORANGE),
]:
    precision, recall, _ = precision_recall_curve(y_test, scores)
    ax.plot(recall, precision, color=color, label=label)
ax.axhline(y_test.mean(), color=GRAY, linestyle="--", linewidth=1)
ax.text(0.98, y_test.mean() + 0.02, "hasard", ha="right", color=GRAY)
ax.set_xlabel("Rappel (part des tricheurs détectés)")
ax.set_ylabel("Précision (part de vrais tricheurs)")
ax.set_title("Courbe précision-rappel sur le jeu de test")
ax.legend()
plt.show()

La courbe de Sentinel est **au-dessus** de celle de la règle actuelle sur toute la plage : pour un même nombre de tricheurs détectés, les modérateurs examinent beaucoup moins de joueurs honnêtes.

In [ ]:
flagged = test_scores >= result.threshold
confusion = pd.crosstab(
    pd.Series(y_test, name="Réalité").map(HONEST_CHEATER_LABELS),
    pd.Series(flagged, name="Décision du modèle").map({False: "Non signalé", True: "Signalé"}),
)
print(f"Seuil de décision retenu : {result.threshold:.3f}")
confusion

**Le seuil :** le modèle donne une probabilité ; le seuil la transforme en décision « à examiner ». Il a été choisi sur la **validation** comme le plus bas seuil gardant au plus 1 % d'honnêtes signalés à tort : on plafonne d'abord le préjudice (être soupçonné à tort), puis on détecte le plus de tricheurs possible.

Rappel : un signalement n'est **jamais** une sanction. Chaque dossier est examiné par un modérateur (article 22 du RGPD).

In [ ]:
result.acceptance

## Conclusions
- Les seuils d'acceptation du cahier des charges sont atteints, y compris l'équité.
- Sur les dossiers examinés chaque jour, la précision passe d'environ **30 %** (signalements) à plus de **90 %** (Sentinel).
- Limites : données synthétiques ; ~30 % des tricheurs restent indétectables par construction (triche intermittente et discrète) ; les métriques varient selon la graine (PR-AUC 0,52 à 0,62 sur 5 graines).
- Le modèle versionné en production est produit par `python -m src.train` (voir `models/cheat_model_v1.json`).